# DramaWiki (wiki.d-addicts.com): Raw HTML Downloader

### Kailyn

### This is an **old** notebook that is no longer being used in the data collection process.

This notebook only downloads and saves the raw HTML files. Parsing will be done in another notebook.

For each selected show (see `mdl_music_kdrama_cleaned.csv`), this notebook saves three kinds of pages locally:
1. the **show page** (e.g. `wiki.d-addicts.com/Dream_High`)
2. the **OST page(s)** linked from the show page (e.g. `.../Dream_High_OST`)
3. the **actor pages for the main cast**, i.e. the people listed under *Main Cast* in the show page's *Cast* section

Everything lands in `raw_html/` (one subfolder per page type) and every request is logged in `raw_html/manifest.csv`.

I'm trying using requests right now, with Selenium as a fallback. The fallback code was written with the help of genAI.

**Table of Contents**

1. [Setup](#1)
2. [Configure Parameters](#2)
3. [Setting up Caching](#3)
4. [Setting up Parsing](#4)
5. [Extract Tag Ids](#5)
6. [Collect Candidate Dramas](#6)
7. [Add Drama Metadata](#7)
8. [Run Everything](#8)
9. [Summary and Save](#9)

<a id=1></a>
## 1. Setup

In [17]:
import csv
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, unquote, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

<a id=2></a>
## 2. Configure Parameters

In [ ]:
WIKI_BASE = "https://wiki.d-addicts.com"

SELECTED_SHOWS = [ # Manual run through #2 - first run-through returned only 50/123 dramas, and manually checking the remaining 70 for sites only returned the following 9
    {"title": "Hospital Playlist Season 2", "wiki_page": "Hospital_Playlist"},
    {"title": "Shut Up: Flower Boy Band", "wiki_page": "Shut_Up_Flower_Boy_Band"},
    {"title": "I'm Sorry, I Love You", "wiki_page": "Sorry_I_Love_You"},
    {"title": "Naeil's Cantabile", "wiki_page": "Nae_Il%27s_Cantabile"},
    {"title": "Drama Stage Season 4: Mint Condition", "wiki_page": "Mint_Condition"},
    {"title": "Bravo My Life", "wiki_page": "Bravo_My_Life_(2017)"},
    {"title": "Monstar", "wiki_page": "Monstar_(tvN)"},
    {"title": "Dream High Season 2", "wiki_page": "Dream_High"},
    {"title": "You're Too Much", "wiki_page": "You Are Too Much"},
]

LOAD_SHOWS_FROM_CSV = False
SHOWS_CSV = "(old) mdl_music_kdrama_cleaned.csv"

MAIN_CAST_LIMIT = None # None = everyone under the "Main Cast" label, but you can set an int to cap per season in case there's a lot of actors

# Politeness
DELAY_SECONDS = 2.0       # minimum pause after every real network request
# DELAY_JITTER = 1.0        # plus a random 0..1s so requests aren't metronomic
MAX_ATTEMPTS = 3          # retries for 429/5xx/odd responses
BACKOFF_SECONDS = 15      # waits 15s, 30s, ... between retries
USER_AGENT = "Mozilla/5.0 (course research scraper)" # unsure if needed

# Fallback to Selenium
USE_SELENIUM_FALLBACK = True   # only launched if requests gets a non-wiki response

# Output Folders
RAW_DIR = Path("raw_html")
for sub in ("shows", "osts", "actors"):
    (RAW_DIR / sub).mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RAW_DIR / "manifest.csv"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

<a id=3></a>
## 3. Fetching and saving pages

All requests go through `fetch_page(url, dest)`.

- **Already saved?** Skip it (no request, no delay). Can also delete a file or pass `force=True` to refresh it (similar to m3_1)
- **Delay:** `time.sleep` after *every* real request (`DELAY_SECONDS`).
- **A missing page (HTTP 404)** is recorded as "page does not exist" and not retried. (Will have to manually go through these)
- **429 / 5xx / a page that doesn't look like a wiki page** is retried with a growing wait, then (if enabled) handed to SeleniumBase.
- Every attempt returns a record that goes into the manifest.

In [ ]:
def polite_sleep():
    """Pause between real requests."""
    time.sleep(DELAY_SECONDS)

def looks_like_wiki_page(html):
    """True if this is a real DramaWiki content page (not a block/challenge page)."""
    return 'id="mw-content-text"' in html

_driver = None   # SeleniumBase browser, created only if the fallback is needed

def fetch_with_selenium(url):
    """Fallback: load the page in a real browser and return its HTML (or None)."""
    global _driver
    if _driver is None:
        from seleniumbase import Driver          # imported lazily: not needed otherwise
        _driver = Driver(page_load_strategy="eager")
    _driver.open(url)
    _driver.sleep(DELAY_SECONDS + 1)
    html = _driver.page_source
    return html if looks_like_wiki_page(html) else None

def close_driver():
    global _driver
    if _driver is not None:
        _driver.quit()
        _driver = None

def fetch_page(url, dest, force=False):
    """Download `url` into the file `dest` (unless already saved).
    Returns a dict describing what happened, for the manifest."""
    dest = Path(dest)
    rec = {"url": url, "path": str(dest), "status": "", "method": "", "note": "",
           "fetched_at": ""}

    if dest.exists() and not force:
        rec.update(status="cached", method="cache")
        return rec

    html_bytes = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            resp = session.get(url, timeout=30)
        except requests.RequestException as e:
            rec["note"] = f"request error: {e}"
            polite_sleep()
            continue

        polite_sleep() # be polite!
        rec["status"] = resp.status_code

        if resp.status_code == 404:
            rec["note"] = "page does not exist"
            return rec # no point retrying or using a browser
        if resp.status_code == 200 and looks_like_wiki_page(resp.text):
            html_bytes, rec["method"] = resp.content, "requests"
            break

        rec["note"] = f"unexpected response (HTTP {resp.status_code}), attempt {attempt}"
        time.sleep(BACKOFF_SECONDS * attempt)

    if html_bytes is None and USE_SELENIUM_FALLBACK:
        print("  requests failed; trying the browser fallback...")
        html = fetch_with_selenium(url)
        if html:
            html_bytes, rec["method"] = html.encode("utf-8"), "selenium"
            rec["status"], rec["note"] = 200, "saved via selenium fallback"

    if html_bytes is None:
        rec["note"] = rec["note"] or "failed"
        return rec

    dest.write_bytes(html_bytes)
    rec["fetched_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return rec

In [20]:
def wiki_url(page_name):
    """'Dream_High' -> 'https://wiki.d-addicts.com/Dream_High' (encoding parens etc.)."""
    return f"{WIKI_BASE}/{quote(page_name.replace(' ', '_'), safe='_')}"


def page_name_from_url(url):
    """'.../Kim_Soo_Hyun_%281988%29' -> 'Kim_Soo_Hyun_(1988)'"""
    return unquote(urlparse(url).path.lstrip("/"))


def local_path(kind, url):
    """Where a page is stored: raw_html/<kind>/<url-encoded page name>.html
    (kind is 'shows', 'osts' or 'actors')."""
    return RAW_DIR / kind / (quote(page_name_from_url(url), safe="") + ".html")

<a id=4></a>
## 4. Finding the OST and main-cast links in a saved show page

From the saved file, we need to pull the OST and cast links. I'm using beautifulsoup.

From my scan-through of the Dream High page, it looks like the following is true:
- One show page can hold **several seasons** (Dream High has Season 1 and Season 2), each with its own *Cast* section and its own *Original Soundtrack* link. I handle every season.
- Under *Cast* there are labeled groups: **Main Cast**, Kirin's Students, Cameos, Extended Cast... I take only the list under the **Main Cast** label.
- Some entries have a **nested** sub-item (e.g. a child actor playing the young version of a role). I keep only the top-level entry.
- Some names are **red links** (`...redlink=1`, "page does not exist"): there's no page to download, so they're skipped.
- Some actors' page names include a **disambiguator**, like `Kim_Soo_Hyun_(1988)`.

If a season has no "Main Cast" label, I fall back to the first list in its Cast section and note it in the output so you can check.

In [21]:
def normalize_wiki_link(href):
    """Return a clean absolute URL if `href` points to a real DramaWiki content
    page; otherwise None (red links, edit links, other sites, File:/Category: pages)."""
    url = urljoin(WIKI_BASE + "/", href)
    p = urlparse(url)
    if p.netloc.lower() != "wiki.d-addicts.com":
        return None
    if p.path.startswith("/index.php") or "redlink" in p.query or "action=" in p.query:
        return None
    name = unquote(p.path.lstrip("/"))
    if not name or ":" in name:
        return None
    return f"{WIKI_BASE}{p.path}"

def extract_ost_links(soup):
    """Links next to an 'Original Soundtrack:' label. One per season, typically."""
    urls = []
    for label in soup.find_all(["b", "strong"]):
        if label.get_text(strip=True).lower().startswith("original soundtrack"):
            container = label.find_parent("li") or label.parent
            for a in container.find_all("a"):
                url = normalize_wiki_link(a.get("href", ""))
                if url and url not in urls:
                    urls.append(url)
    return urls

def extract_main_cast_links(soup):
    """Return [{'name', 'url', 'season', 'note'}] for the 'Main Cast' list in
    every 'Cast' section of a show page."""
    results = []
    cast_headings = [
        h for h in soup.find_all(["h2", "h3", "h4"])
        if re.match(r"^\s*Cast\b", h.get_text(" ", strip=True))
    ]

    for season_no, heading in enumerate(cast_headings, start=1):
        # The Cast section = everything after its heading up to the next h2/h3.
        section = []
        for sib in heading.next_siblings:
            name = getattr(sib, "name", None)
            if name in ("h2", "h3"):
                break
            if name:
                section.append(sib)

        # Find the "Main Cast" label, then the first <ul> after it.
        marker = next(
            (i for i, el in enumerate(section)
             if el.name != "ul"
             and el.get_text(" ", strip=True).lower().rstrip(":") == "main cast"),
            None,
        )
        note = "" if marker is not None else "no 'Main Cast' label; used first list"
        start = 0 if marker is None else marker + 1
        ul = next((el for el in section[start:] if el.name == "ul"), None)
        if ul is None:
            continue

        count = 0
        for li in ul.find_all("li", recursive=False):      # top-level entries only
            # first link that belongs to THIS li, not to a nested sub-list
            a = next((a for a in li.find_all("a") if a.find_parent("ul") is ul), None)
            if a is None:
                continue
            url = normalize_wiki_link(a.get("href", ""))
            if url is None:                                  # red link / off-wiki
                continue
            results.append({"name": a.get_text(strip=True), "url": url,
                            "season": season_no, "note": note})
            count += 1
            if MAIN_CAST_LIMIT and count >= MAIN_CAST_LIMIT:
                break
    return results

<a id=5></a>
## 5. Downloading for every show

In [22]:
def resolve_shows():
    """Build the list of shows from SELECTED_SHOWS or (optionally) the CSV."""
    if not LOAD_SHOWS_FROM_CSV:
        return SELECTED_SHOWS
    df = pd.read_csv(SHOWS_CSV)
    # df = df.head(3) # Test run, only first three dramas
    return [{"title": t, "wiki_page": None} for t in df["title"]]

def download_everything(shows):
    manifest = []

    def log(show_title, kind, extra, rec):
        manifest.append({"show": show_title, "kind": kind, **extra, **rec})
        print(f"    {kind:6} {rec['status']!s:7} {rec['url']}  {rec['note']}")

    for show in shows:
        title = show["title"]
        page = show.get("wiki_page") or title.replace(" ", "_")
        show_url = wiki_url(page)
        print(f"\n=== {title} ===")

        # 1) the show page
        rec = fetch_page(show_url, local_path("shows", show_url))
        log(title, "show", {}, rec)
        if not Path(rec["path"]).exists():
            print("    -> couldn't get the show page, skipping its OST/cast pages")
            continue

        soup = BeautifulSoup(Path(rec["path"]).read_bytes(), "html.parser")

        # 2) the OST page(s)
        ost_urls = extract_ost_links(soup)
        if not ost_urls:
            print("    (no 'Original Soundtrack' link found on the show page)")
        for ost_url in ost_urls:
            rec = fetch_page(ost_url, local_path("osts", ost_url))
            log(title, "ost", {}, rec)

        # 3) the main-cast actor pages
        cast = extract_main_cast_links(soup)
        print(f"    found {len(cast)} main-cast links")
        for member in cast:
            rec = fetch_page(member["url"], local_path("actors", member["url"]))
            log(title, "actor", {"actor_name": member["name"],
                                 "season": member["season"],
                                 "cast_note": member["note"]}, rec)
    return manifest

<a id=6></a>
## 6. Run it

In [ ]:
manifest = []
try:
    manifest = download_everything(resolve_shows())
finally:
    close_driver()                           # shut the browser down if it was ever opened
    if manifest:                             # write the log even if the run crashed
        pd.DataFrame(manifest).to_csv(MANIFEST_PATH, index=False, mode="a")
        print(f"\nManifest written to {MANIFEST_PATH}")


=== Hospital Playlist Season 2 ===
    show   cached  https://wiki.d-addicts.com/Hospital_Playlist  
    ost    cached  https://wiki.d-addicts.com/Hospital_Playlist_OST  
    ost    cached  https://wiki.d-addicts.com/Hospital_Playlist_2_OST  
    found 0 main-cast links

=== Shut Up: Flower Boy Band ===
    show   200     https://wiki.d-addicts.com/Shut_Up_Flower_Boy_Band  
    ost    200     https://wiki.d-addicts.com/Shut_Up_Flower_Boy_Band_OST  
    found 6 main-cast links
    actor  200     https://wiki.d-addicts.com/Sung_Joon_(Bang_Sung_Joon)_(1990)  
    actor  cached  https://wiki.d-addicts.com/L  
    actor  200     https://wiki.d-addicts.com/Lee_Hyun_Jae_(1988)  
    actor  200     https://wiki.d-addicts.com/Kim_Min_Suk  
    actor  200     https://wiki.d-addicts.com/Yoo_Min_Kyu  
    actor  200     https://wiki.d-addicts.com/Lee_Min_Ki_(1985)  

=== I'm Sorry, I Love You ===
    show   200     https://wiki.d-addicts.com/Sorry_I_Love_You  
    ost    200     https://wiki.d-ad

<a id=7></a>
## 7. Summary

In [24]:
m = pd.DataFrame(manifest)

if m.empty:
    print("Nothing in the manifest.")
else:
    m["outcome"] = m["status"].astype(str)
    print("Pages by type and outcome:")
    print(m.groupby(["kind", "outcome"]).size().unstack(fill_value=0), "\n")

    problems = m[(m["note"] != "") & (m["outcome"] != "cached")]
    print(f"{len(problems)} pages with notes/problems:")
    display(problems[["show", "kind", "url", "status", "note"]]) if len(problems) else print("  none")

    print("\nFiles on disk:")
    for sub in ("shows", "osts", "actors"):
        print(f"  raw_html/{sub}/: {len(list((RAW_DIR / sub).glob('*.html')))} files")

    print("\nRAW PAGES DOWNLOADED. Stop here and check them before writing parsers.")

Pages by type and outcome:
outcome  200  cached
kind                
actor     41       3
ost        6       4
show       7       2 

1 pages with notes/problems:


,show,kind,url,status,note
27,Naeil's Cantabile,show,https://wiki.d-addicts.com/Nae_Il%2527s_Cantabile,200,saved via selenium fallback



Files on disk:
  raw_html/shows/: 58 files
  raw_html/osts/: 46 files
  raw_html/actors/: 247 files

RAW PAGES DOWNLOADED. Stop here and check them before writing parsers.
